# Visualisation of LFP & Sleep scoring

### Import recordings

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import signal
import os
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from ephyviewer import mkQApp, MainViewer, TraceViewer
import ephyviewer
from scipy.stats import zscore
from itertools import groupby
from ephyviewer import mkQApp, MainViewer, TraceViewer, TimeFreqViewer
from ephyviewer import InMemoryAnalogSignalSource
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, EpochEncoder
import json
import IPython
from matplotlib import cm
from matplotlib.colors import to_hex
import ast
from ephyviewer import AnalogSignalSourceWithScatter
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
from scipy import stats
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from datetime import datetime, timezone
from zoneinfo import ZoneInfo  # Only in Python 3.9+
import shutil
import mne

%matplotlib widget

Choose OpenEphys folder

In [235]:
dpath = "C:/Users/Manip2/Documents/EpiKid/transfer_11012822_files_c1d4dc1b/AGAL06-E_06032017.edf"
folder_base = Path(dpath) 

In [237]:
txtt = "C:/Users/Manip2/Documents/EpiKid/transfer_11012822_files_c1d4dc1b/AGAL06-E_06032017_test.txt"
with open(txtt, encoding="utf_8", errors='ignore') as f:
    contents = f.read()
print(contents)
keep_words = ["Stage N1  ", "Stage N2  ", "Stage N3  ", "REM  ", "WAKE  "]
s = contents
matches = []
i = 0
while i < len(s):
    for w in keep_words:
        if s.startswith(w, i):
            matches.append(w)
            i += len(w)
            break
    else:
        i += 1  # move forward if no match

  
 CFileHeader Startdate 06-mars-2017
Electrode transducer types:
Fp1 : Unknown
TB1 : Unknown
Fp2 : Unknown
F7 : Unknown
F3 : Unknown
Fz : Unknown
F4 : Unknown
F8 : Unknown
T3 : Unknown
C3 : Unknown
Cz : Unknown
C4 : Unknown
T4 : Unknown
T5 : Unknown
P3 : Unknown
Pz : Unknown
P4 : Unknown
T6 : Unknown
O1 : Unknown
TB2 : Unknown
O2 : Unknown
Roc : Unknown
Loc : Unknown
EMG1_ : Unknown
EMG2_ : Unknown
C~1 : Unknown
Ment_ : Unknown
PNG3_ : Unknown
BEAT_ : Unknown
SpO2_ : Unknown
PULS_ : Unknown  Qd-BHarmonie 7.0a Release ENHarmonie 7.0a Release EN$003bab5f-9b93-4069-a66b-cb23dbca9d07  CPatientInfo Andre   /+-B         &Andre Gomes Moreno Alicia 04-nov.-2004       	 CStdCoDec  @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @ @   CCalibration       p@     ?                                                                 H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H  H           ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  ?  

In [272]:
data = mne.io.read_raw_edf(dpath, encoding='latin1')
raw_data = data.get_data()
# you can get the metadata included in the file and a list of all channels:
info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz

print(data.info.get('meas_date'))
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

Extracting EDF parameters from C:\Users\Manip2\Documents\EpiKid\transfer_11012822_files_c1d4dc1b\AGAL06-E_06032017.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...


C:\Users\Manip2\AppData\Local\Temp\ipykernel_28588\2517313479.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  data = mne.io.read_raw_edf(dpath, encoding='latin1')


2017-03-06 20:57:13+00:00
10.601h recording if sampled at 256.0Hz (38164.0 seconds)
31 channels found: ['EEG Fp1', 'EEG TB1', 'EEG Fp2', 'EEG F7', 'EEG F3', 'EEG Fz', 'EEG F4', 'EEG F8', 'EEG T3', 'EEG C3', 'EEG Cz', 'EEG C4', 'EEG T4', 'EEG T5', 'EEG P3', 'EEG Pz', 'EEG P4', 'EEG T6', 'EEG O1', 'EEG TB2', 'EEG O2', 'EEG Roc', 'EEG Loc', 'EMG1+', 'EMG2+', '.....', 'Ment+', 'PNG3+', 'BEAT+', 'SpO2+', 'PULS+']


In [258]:
print('Does the number of detected epoch match the length of the signal?', np.floor(duration/30) == len(matches))
SleepScoring= pd.DataFrame(matches, columns=['label'])
SleepScoring['time'] = list(range(0, len(SleepScoring) * 30, 30))
SleepScoring['duration'] = 30
# correct if not matching total recording time
SleepScoring.loc[(SleepScoring.shape[0]-1), 'duration']= duration - (SleepScoring.loc[(SleepScoring.shape[0]-2), 'time'] +30)
AutoSleepScoring_filename = os.path.join(Path(f'{Path(txtt).parent}'), f'EphyViewer_sts_approxScoring.csv')
SleepScoring.to_csv(AutoSleepScoring_filename, index=False)

Does the number of detected epoch match the length of the signal? False


In [137]:
montage = 'Fp2-F4,F4-C4,C4-P4,P4-O2,Fp1-F3,F3-C3,C3-P3,P3-O1,Fz-Cz,Cz-Pz,Fp2-F8,F8-T4,T4-T6,T6-O2,Fp1-F7,F7-T3,T3-T5,T5-O1,EMG1+,EMG2+'
channels_clean = [ch.replace('EEG ', '') for ch in channels]

pairs = montage.split(',')
montage_eeg=[]
montage_name=[]
for pair in pairs:
    if pair.count('-')==0:
        ch1 = pair
        idx1 = channels_clean.index(ch1)
        eeg=combined[:,idx1]
        #montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        #montage_name.append(pair)
    else:        
        ch1, ch2 = pair.split('-')
        idx1 = channels_clean.index(ch1)
        idx2 = channels_clean.index(ch2)
        eeg=combined[:,idx2]-combined[:,idx1]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(pair)

Check all channels

In [138]:
app = mkQApp()
winlen = 20 # default window length in sec
t_start = 0.

win = MainViewer(debug=False, show_auto_scale=True)
view1 = TraceViewer.from_numpy(montage_eeg, samplerate, t_start, 'Signals', channel_names=montage_name)

cmap = cm.Spectral
values = np.linspace(0, 1, np.array(montage_eeg).shape[1])
colormap = [to_hex(rgb) for rgb in cmap(values)]
for idx in range(np.array(montage_eeg).shape[1]): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx]

view1.params['display_labels'] = True
view1.params['scale_mode'] = 'same_for_all'
view1.auto_scale()
view1.params['xsize'] = winlen
win.add_view(view1)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)

#Run
win.show()
app.exec()


0

In [277]:
#NEED TO APPLY A FILTER TO THE EEG SIGNALS (LOWPASS 70HZ, HIGHPASS 0.5HZ)

With sleep scoring

In [276]:
labels = ["WAKE  ", "Stage N1  ", "Stage N2  ", "Stage N3  ", "REM  ", ]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{Path(txtt).parent}/EphyViewer_sts_approxScoring.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=montage_name)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 30
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#ffaa00" 
view2.by_label_params['label1', 'color'] = '#69ffe6'
view2.by_label_params['label2', 'color'] = '#69cfff'
view2.by_label_params['label3', 'color'] = '#cd69ff'
view2.by_label_params['label4', 'color'] = "#ff00aa"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'


view2.controls.hide()

win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

0